# Tools with MCP ⏰

The Model Context Protocol (MCP) provides a standardized way to connect AI agents to external tools and data sources. Let's connect to an MCP server using `langchain-mcp-adapters`.

In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd()
while project_root != project_root.parent and not (
    project_root / "utils" / "env_utils.py"
).is_file():
    project_root = project_root.parent

if not (project_root / "utils" / "env_utils.py").is_file():
    raise FileNotFoundError("Could not locate utils/env_utils.py from the notebook directory.")

os.chdir(project_root)
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
from utils.env_utils import doublecheck_env, doublecheck_pkgs

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")  # check environmental variables
doublecheck_pkgs(pyproject_path="pyproject.toml", verbose=True)   # check packages

Did not find file example.env.
This is used to double check the key settings for the notebook.
This is just a check and is not required.

Python 3.13.5 satisfies requires-python: >=3.11, <3.14
package                | required | installed | status | path                                                                            
---------------------- | -------- | --------- | ------ | --------------------------------------------------------------------------------
jupyter                | >=1.0.0  | 1.1.1     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/python3.13/site-packages
langchain              | >=1.3.10 | 1.4.3     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/python3.13/site-packages
langchain-anthropic    | >=1.4.6  | 1.7.5     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/python3.13/site-packages
langchain-community    | >=0.4.2  | 0.4.2     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/pyth

In [2]:
import sys

# MCP's Windows subprocess fallback needs a real stderr file handle.
# Jupyter replaces sys.stderr with an object whose fileno() is unsupported.
if sys.platform == "win32":
    sys.stderr = sys.__stderr__

from langchain_mcp_adapters.client import MultiServerMCPClient
import shutil
import tempfile
from pathlib import Path

# Use the official Python MCP time server. The npm server previously used here
# does not install on Windows. Resolving uvx gives subprocess an absolute path,
# which also avoids the Windows uvx.exe/command lookup difference.

uvx = shutil.which("uvx")
if uvx is None:
    raise RuntimeError("uvx was not found. Install uv from https://docs.astral.sh/uv/.")

mcp_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": uvx,
            "args": [
                "--cache-dir",
                str(Path(tempfile.gettempdir()) / "mcp-uv-cache"),
                # mcp-server-time (mcp>=1.23.0, no upper bound) breaks against
                # mcp 2.0.0+, which renamed McpError to MCPError:
                # https://github.com/modelcontextprotocol/servers/issues/4570
                "--with",
                "mcp<2",
                "mcp-server-time",
            ],
        }
    }
)

# Load tools from the MCP server
mcp_tools = await mcp_client.get_tools()
print(f"Loaded {len(mcp_tools)} tools from the MCP tools: {[t.name for t in mcp_tools]}")

Loaded 2 tools from the MCP tools: ['get_current_time', 'convert_time']


Create an agent with the MCP-provided time tools.

In [3]:
from langchain.agents import create_agent

agent_with_mcp = create_agent(
    model="openai:gpt-5",
    tools=mcp_tools,
    system_prompt="You are a helpful assistant",
)

Ask about the current time in San Francisco.

In [4]:
result =  await agent_with_mcp.ainvoke(
    {"messages": [{"role": "user", "content": "What is the time in SF right now?"}]}
)
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is the time in SF right now?
================================== Ai Message ==================================
Tool Calls:
  get_current_time (call_zdNk1XrNCobfqygQ9qagSTCB)
 Call ID: call_zdNk1XrNCobfqygQ9qagSTCB
  Args:
    timezone: America/Los_Angeles
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "America/Los_Angeles",\n  "datetime": "2026-10-12T15:47:19-07:00",\n  "day_of_week": "Monday",\n  "is_dst": true\n}', 'id': 'lc_747dac75-762a-4f57-b541-990b5aea6adf'}]
================================== Ai Message ==================================

It’s Monday, 3:47 PM in San Francisco (America/Los_Angeles).
